# 📝 Week 5 Part 1: NLP Basics for ML Interviews

## Overview
NLP is a common interview topic. You need to understand the fundamentals, not just transformers.

## 🎯 Learning Objectives
1. Master text preprocessing techniques
2. Understand TF-IDF and when to use it
3. Build a simple text classifier (spam detection)
4. Know when to use traditional vs deep learning methods

## 💡 Interview Insight
Interviewers often ask: "Walk me through how you'd build a spam classifier without using transformers."

---

In [ ]:
# ============================================================
# IMPORTS AND SETUP
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# NLP imports
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Sklearn for ML
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Download NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)

RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-whitegrid')

print("Setup complete!")

---
## 1. Text Preprocessing Pipeline

### Interview Question: "Walk me through your text preprocessing steps"

Answer framework:
1. Lowercasing
2. Removing noise (HTML, special chars)
3. Tokenization
4. Stopword removal
5. Stemming/Lemmatization

**Key insight:** The order matters, and you should justify each step!

In [ ]:
# ============================================================
# TEXT PREPROCESSING FUNCTIONS
# ============================================================

def preprocess_text(text, 
                    lowercase=True,
                    remove_punctuation=True, 
                    remove_stopwords=True,
                    lemmatize=True):
    """
    Complete text preprocessing pipeline.
    
    Args:
        text: Raw input text
        lowercase: Convert to lowercase
        remove_punctuation: Remove punctuation marks
        remove_stopwords: Remove common words
        lemmatize: Apply lemmatization
    
    Returns:
        Preprocessed text string
    """
    if not isinstance(text, str):
        return ""
    
    # Step 1: Lowercase
    if lowercase:
        text = text.lower()
    
    # Step 2: Remove URLs and email addresses
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text)
    
    # Step 3: Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Step 4: Remove numbers (optional - depends on use case)
    text = re.sub(r'\d+', '', text)
    
    # Step 5: Remove punctuation
    if remove_punctuation:
        text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Step 6: Tokenize
    tokens = word_tokenize(text)
    
    # Step 7: Remove stopwords
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [t for t in tokens if t not in stop_words]
    
    # Step 8: Lemmatization (or stemming)
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    # Step 9: Remove short tokens
    tokens = [t for t in tokens if len(t) > 2]
    
    return ' '.join(tokens)


# Example
sample_text = """Hello! Check out this AMAZING offer at http://spam.com!!!
You've WON $1,000,000 dollars!!! <b>Act now!</b> email: spam@fake.com"""

print("Original text:")
print(sample_text)
print("\nPreprocessed text:")
print(preprocess_text(sample_text))

In [ ]:
# ============================================================
# STEMMING VS LEMMATIZATION - INTERVIEW QUESTION!
# ============================================================

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ['running', 'runs', 'ran', 'better', 'studies', 'studying', 'mice']

print("Stemming vs Lemmatization Comparison:")
print("="*50)
print(f"{'Word':<12} {'Stemmed':<12} {'Lemmatized':<12}")
print("-"*50)

for word in words:
    stemmed = stemmer.stem(word)
    lemmatized = lemmatizer.lemmatize(word, pos='v')  # verb POS
    print(f"{word:<12} {stemmed:<12} {lemmatized:<12}")

print("\n📌 Interview Answer:")
print("- Stemming: Faster, crude, rule-based (chops off endings)")
print("- Lemmatization: Slower, accurate, uses vocabulary/dictionary")
print("- Use lemmatization when meaning matters (sentiment analysis)")
print("- Use stemming for speed/information retrieval")

---
## 2. Text Vectorization: Bag of Words & TF-IDF

### Interview Question: "Explain TF-IDF and when you'd use it over word embeddings"

In [ ]:
# ============================================================
# BAG OF WORDS (CountVectorizer)
# ============================================================

sample_docs = [
    "I love machine learning",
    "Machine learning is great",
    "I love deep learning too"
]

# Count Vectorizer
count_vec = CountVectorizer()
bow_matrix = count_vec.fit_transform(sample_docs)

print("Bag of Words Matrix:")
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=count_vec.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(sample_docs))]
)
print(bow_df)

print("\n📌 BoW Pros/Cons:")
print("+ Simple and interpretable")
print("+ Fast to compute")
print("- Ignores word order")
print("- Common words dominate (solved by TF-IDF)")

In [ ]:
# ============================================================
# TF-IDF (Term Frequency - Inverse Document Frequency)
# ============================================================

tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(sample_docs)

print("TF-IDF Matrix:")
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf_vec.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(sample_docs))]
)
print(tfidf_df)

print("\n📌 TF-IDF Formula:")
print("TF(t,d) = count(t in d) / total words in d")
print("IDF(t) = log(N / docs containing t)")
print("TF-IDF = TF × IDF")

print("\n📌 Why TF-IDF?")
print("- Downweights common words (the, is, a)")
print("- Upweights rare, distinctive words")
print("- Still interpretable unlike embeddings")

In [ ]:
# ============================================================
# WHEN TO USE WHAT - INTERVIEW CHEAT SHEET
# ============================================================

print("📋 Text Vectorization Decision Guide:")
print("="*60)
print("""
Use TF-IDF when:
  ✓ Small dataset (< 10k samples)
  ✓ Need interpretability
  ✓ Domain-specific vocabulary
  ✓ Fast inference required
  ✓ Limited compute resources
  
Use Word Embeddings (Word2Vec, GloVe) when:
  ✓ Need semantic similarity
  ✓ Words with similar meanings should be grouped
  ✓ Medium-sized datasets
  
Use Transformers (BERT, etc.) when:
  ✓ Large dataset available
  ✓ Need context understanding
  ✓ State-of-the-art performance required
  ✓ Have GPU resources
""")

---
## 3. Building a Spam Classifier

### Classic interview exercise: Build an end-to-end text classifier

In [ ]:
# ============================================================
# CREATE SYNTHETIC SPAM DATASET
# ============================================================

np.random.seed(RANDOM_STATE)

# Spam patterns
spam_templates = [
    "Congratulations! You've won {} dollars! Click here now!",
    "URGENT: Your account will be suspended. Verify at {}",
    "Free {} for the first 100 customers! Limited time!",
    "You're a winner! Claim your {} prize today!",
    "Act now! {} off all products! This weekend only!",
    "Your {} has been selected for a special offer!",
    "Make money fast! Earn {} working from home!",
    "Hot singles in your area! Meet {} today!",
    "Cheap {} pills! No prescription needed!",
    "Investment opportunity! Guaranteed {} return!",
]

ham_templates = [
    "Hi, can we schedule a meeting for {} next week?",
    "Please find the {} report attached.",
    "Thanks for your help with the {} project.",
    "Reminder: {} meeting at 3pm today.",
    "I'll review the {} and get back to you.",
    "Looking forward to discussing {} tomorrow.",
    "Can you send me the {} files?",
    "Great presentation on {}! Well done.",
    "Let's catch up about {} over coffee.",
    "The {} deadline has been extended to Friday.",
]

fillers = ['amazing', 'special', 'exclusive', 'limited', 'important', 
           'quarterly', 'budget', 'strategy', 'marketing', 'sales']

n_samples = 2000
texts = []
labels = []

for _ in range(n_samples // 2):
    # Spam
    template = np.random.choice(spam_templates)
    filler = np.random.choice(['$10,000', '$50,000', '1,000,000', 'FREE', '50%', '75%'])
    texts.append(template.format(filler))
    labels.append(1)
    
    # Ham
    template = np.random.choice(ham_templates)
    filler = np.random.choice(fillers)
    texts.append(template.format(filler))
    labels.append(0)

df_spam = pd.DataFrame({'text': texts, 'label': labels})
df_spam = df_spam.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Dataset shape: {df_spam.shape}")
print(f"\nLabel distribution:")
print(df_spam['label'].value_counts())
print("\nSample messages:")
for label in [0, 1]:
    print(f"\n{'Ham' if label == 0 else 'Spam'}:")
    print(df_spam[df_spam['label'] == label]['text'].iloc[0])

In [ ]:
# ============================================================
# TRAIN/TEST SPLIT
# ============================================================

X = df_spam['text']
y = df_spam['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# ============================================================
# BUILD AND COMPARE MODELS
# ============================================================

# Define pipelines
pipelines = {
    'Naive Bayes + BoW': Pipeline([
        ('vectorizer', CountVectorizer(max_features=5000)),
        ('classifier', MultinomialNB())
    ]),
    
    'Naive Bayes + TF-IDF': Pipeline([
        ('vectorizer', TfidfVectorizer(max_features=5000)),
        ('classifier', MultinomialNB())
    ]),
    
    'Logistic Regression + TF-IDF': Pipeline([
        ('vectorizer', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    
    'Random Forest + TF-IDF': Pipeline([
        ('vectorizer', TfidfVectorizer(max_features=5000)),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
    ])
}

# Train and evaluate
results = {}

print("Model Comparison:")
print("="*60)

for name, pipeline in pipelines.items():
    # Cross-validation
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1')
    
    # Fit and predict
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    # Metrics
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'pipeline': pipeline,
        'cv_f1': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_auc': auc,
        'y_pred': y_pred
    }
    
    print(f"{name}:")
    print(f"  CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
    print(f"  Test AUC: {auc:.4f}")
    print()

In [ ]:
# ============================================================
# BEST MODEL EVALUATION
# ============================================================

best_name = max(results.keys(), key=lambda k: results[k]['test_auc'])
best_pipeline = results[best_name]['pipeline']
best_pred = results[best_name]['y_pred']

print(f"Best Model: {best_name}")
print("="*60)
print("\nClassification Report:")
print(classification_report(y_test, best_pred, target_names=['Ham', 'Spam']))

# Confusion Matrix
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title(f'Confusion Matrix - {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FEATURE IMPORTANCE (TOP SPAM/HAM WORDS)
# ============================================================

if 'Logistic Regression' in best_name:
    vectorizer = best_pipeline.named_steps['vectorizer']
    classifier = best_pipeline.named_steps['classifier']
    
    feature_names = vectorizer.get_feature_names_out()
    coefficients = classifier.coef_[0]
    
    # Top spam words (positive coefficients)
    top_spam_idx = np.argsort(coefficients)[-10:][::-1]
    
    # Top ham words (negative coefficients)
    top_ham_idx = np.argsort(coefficients)[:10]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Spam words
    spam_words = [feature_names[i] for i in top_spam_idx]
    spam_coefs = [coefficients[i] for i in top_spam_idx]
    axes[0].barh(spam_words, spam_coefs, color='coral')
    axes[0].set_title('Top 10 Spam Indicators')
    axes[0].set_xlabel('Coefficient')
    
    # Ham words
    ham_words = [feature_names[i] for i in top_ham_idx]
    ham_coefs = [coefficients[i] for i in top_ham_idx]
    axes[1].barh(ham_words, ham_coefs, color='steelblue')
    axes[1].set_title('Top 10 Ham Indicators')
    axes[1].set_xlabel('Coefficient')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# PREDICT ON NEW MESSAGES
# ============================================================

new_messages = [
    "You've won a free iPhone! Click here to claim!",
    "Can we reschedule our meeting to Thursday?",
    "URGENT: Your bank account has been compromised!",
    "Thanks for sending the quarterly report."
]

predictions = best_pipeline.predict(new_messages)
probabilities = best_pipeline.predict_proba(new_messages)

print("Predictions on New Messages:")
print("="*60)

for msg, pred, prob in zip(new_messages, predictions, probabilities):
    label = 'SPAM' if pred == 1 else 'HAM'
    confidence = max(prob) * 100
    print(f"\nMessage: {msg[:50]}...")
    print(f"Prediction: {label} ({confidence:.1f}% confidence)")

---
## 4. Common Interview Questions

### Q1: How would you handle out-of-vocabulary words?
**Answer:** 
- TF-IDF: Simply ignores them (won't match any feature)
- Subword tokenization (BPE): Breaks into known pieces
- Character n-grams: `analyzer='char_wb'` in TfidfVectorizer

### Q2: Why does Naive Bayes work well for text?
**Answer:**
- Assumption of feature independence often holds for word presence
- Works well with high-dimensional sparse data
- Fast training and inference
- Often competitive baseline for text classification

### Q3: When would TF-IDF fail?
**Answer:**
- When context matters ("not good" vs "good")
- When semantic similarity is important
- Long documents with diverse topics
- When word order is critical

---
## ✅ Week 5 Part 1 Checklist

- [x] Text preprocessing pipeline
- [x] Understand stemming vs lemmatization
- [x] Implement BoW and TF-IDF
- [x] Build spam classifier
- [x] Compare different models
- [x] Analyze feature importance

---

**Next: Week 5 Part 2 - Model Deployment** 🚀